In [61]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

In [62]:
# Configuración de la conexión
server = '10.0.0.133'  # Reemplaza con el nombre o dirección IP de tu servidor SQL Server
database = 'DB_GENERAL'  # Reemplaza con el nombre de tu base de datos
username = 'sa'  # Reemplaza con tu nombre de usuario
password = 'Essalud23**'  # Reemplaza con tu contraseña
driver = 'ODBC Driver 17 for SQL Server'
# Cadena de conexión
connection_string = f'mssql+pyodbc://{username}:{password}@{server}/{database}?driver={driver}'

In [63]:
try:
    # Establecer la conexión con SQLAlchemy
    engine = create_engine(connection_string)

    # Especificar el esquema
    schema = 'jcardenas'

    # Construir la consulta SQL
    query = f"SELECT * FROM {schema}.H_DEPENDENCIA WHERE CONDICION = 'ACTIVO' AND (UPPER(dependencia) NOT LIKE 'RED %' OR UPPER(dependencia) LIKE '% RED' OR UPPER(dependencia) LIKE '% RED %' OR UPPER(dependencia) = 'RED')"

    # Leer los resultados en un DataFrame de Pandas
    df = pd.read_sql(query, engine)

except Exception as e:
    print(f'Error de conexión: {e}')

finally:
    # No es necesario cerrar la conexión con SQLAlchemy
    pass

In [64]:
df = df.drop(columns=['PISO', 'CONDICION', 'DIRECTO', 'ANEXO', 'SECCION', 'CENTRAL',
       'EMAIL', 'FAX', 'EQUIVALENCIA', 'PRIORIDAD', 'TIPO_DEPENDENCIA',
       'CODDEP_PUB', 'PUBLICADO_WEB', 'USUARIO_CREACION',
       'USUARIO_MODIFICACION', 'ID_TIPO_DEPENDENCIA',
       'PUBLICADO_EXTRANET', 'CODDEP_PRINCIPAL', 'ES_DEPGENERAL',
       'RUC_ENTIDAD', 'coddep_siad', 'codigo_siad', 'siglas_siad',
       'ES_PRINCIPAL', 'id_persona', 'es_mesa_partes', 'AUDIT_CREA','AUDIT_MOD','id_persona','es_mesa_partes','codigo_expediente','codigo_resolucion'])

In [65]:
df.rename(columns={
    'CODIGO_DEPENDENCIA': 'co_dependencia',
    'DEPENDENCIA': 'de_dependencia',
    'SIGLAS': 'de_corta_depen',
    'CODDEP_RESPONSABLE': 'co_depen_padre'
}, inplace=True)

In [66]:
df['de_corta_depen'] = df['de_corta_depen'].apply(lambda x: x.split('-')[0])

In [67]:
df['co_depen_padre'] = df['co_depen_padre'].replace({0: 1}).fillna(1).astype('Int64')

In [68]:
df

,co_dependencia,de_dependencia,de_corta_depen,co_depen_padre
0,1,SEGURO SOCIAL DE SALUD DEL PERÚ,ESSALUD,1
1,2,GERENCIA CENTRAL DE PLANEAMIENTO Y PRESUPUESTO,GCPP,1
2,3,OFICINA DE APOYO Y CONTROL,OAC,2
3,4,GERENCIA DE PRESUPUESTO,GP,2
4,5,SUB GERENCIA DE PROCESO PRESUPUESTAL,SGPP,4
...,...,...,...,...
328,361,Undidad de Atencion y Calificacion - OSPE San ...,UAC,53
329,362,Prestaciones presenciales y virtuales a pacien...,PPVPPADRFF23,143
330,363,Serv relacionado denominado seguimiento a la p...,SRDSPIRCTPA2023,143
331,364,REPRESENTANTE DE ESSALUD JGA - ESVICSAC,REE,286


In [10]:
DB_USER='postgres'
DB_PASSWORD='SdDd3v'
DB_NAME="sgd"
DB_PORT="5433"
DB_HOST='10.0.28.212'
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

In [11]:
df.to_sql(name=f'rhtm_dependencia',schema='idosgd' ,con=engine3, if_exists='append', index=False,chunksize=5000)

294

In [ ]:
df.to_csv('dependencias.xls', index=False)